# Algoritmos de optimización - Seminario

Nombre y Apellidos:   
* Andrés Gómez
* Iván Cañón


Url: https://github.com/Icanongonz/SEMINARIO/blob/colab/Seminario_Algoritmos.ipynb<br>
Problema: **Sesiones de doblaje**

Descripción del problema: Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las
tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de
grabación independientemente del número de tomas que se graben. No es posible grabar más
de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los
servicios de los actores de doblaje sea el menor posible                                        

(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Respuesta: 

* La estructura elegida para los datos del problema es una matriz de *t* tomas x *a* actores, en cada elemento (toma, actor) un *1* o *0* según si el actor tiene participación o no. Es una estructura simple que nos permitirá calcular los gastos en base a los actores involucrados en las tomas.

* La estructura para representar la solución es un array de *t* tomas, donde cada elemento representa el día que ser realizará la misma. Es una estructura que nos permitirá hacer diferentes asignaciones de los días sin complejidad, cambiar los días de una toma, intercambiarlos, etc. Calcular la limitación de máximo 6 tomas por días es una operación sencilla de acumular las ocurrencias de cada día.

Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Respuesta: 

El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. 


<u>Es un problema de minimización</u>

### importación de librerías

In [327]:
import numpy as np
import pandas as pd

### Carga de datos del problema

In [328]:
# recuperar las tomas/actor desde csv
df = pd.read_csv('Datos problema doblaje (30 tomas, 10 actores).csv', delimiter=';')
df = df.drop('Toma', axis=1)
# generar la matriz toma x actor
problema = df.to_numpy()
print(problema)

[[1 1 1 1 1 0 0 0 0 0]
 [0 0 1 1 1 0 0 0 0 0]
 [0 1 0 0 1 0 1 0 0 0]
 [1 1 0 0 0 0 1 1 0 0]
 [0 1 0 1 0 0 0 1 0 0]
 [1 1 0 1 1 0 0 0 0 0]
 [1 1 0 1 1 0 0 0 0 0]
 [1 1 0 0 0 1 0 0 0 0]
 [1 1 0 1 0 0 0 0 0 0]
 [1 1 0 0 0 1 0 0 1 0]
 [1 1 1 0 1 0 0 1 0 0]
 [1 1 1 1 0 1 0 0 0 0]
 [1 0 0 1 1 0 0 0 0 0]
 [1 0 1 0 0 1 0 0 0 0]
 [1 1 0 0 0 0 1 0 0 0]
 [0 0 0 1 0 0 0 0 0 1]
 [1 0 1 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 0]
 [1 0 1 1 1 0 0 0 0 0]
 [0 0 0 0 0 1 0 1 0 0]
 [1 1 1 1 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 0 0 0]
 [1 1 0 1 0 0 0 0 0 1]
 [1 0 1 0 1 0 0 0 1 0]
 [0 0 0 1 1 0 0 0 0 0]
 [1 0 0 1 0 0 0 0 0 0]
 [1 0 0 0 1 1 0 0 0 0]
 [1 0 0 1 0 0 0 0 0 0]]


Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

In [329]:
# costo para penalizar las soluciones que no cumplen la restricción
COSTO_RESTRICCION = 10000

# Calcula el costo de una solución.
def calcular_costo(problema, solucion):
    '''
    Calcula el costo de una solución.
    Si se exceden las 6 tomas en un mismo día, habrá un costo extra COSTO_RESTRICCION para penalizar la solución
    '''
    costo = 0
    
    # la cantidad de dias maxima es igual a la cantidad de tomas (es en el caso de que se haga una toma por dia)
    tomas = len(solucion)

    # agrupar las tomas por dia
    dias = [[] for i in range(tomas)]
    for t in range(tomas):
        d = solucion[t]
        dias[d].append(t)
        
    # calcular el costo por dia
    for d in range(len(dias)):
        # actores que van a participar en las tomas de ese dia
        actores = []
        # recorremos las tomas de ese día 
        for t in dias[d]:
            for a in range(problema.shape[1]):
                if problema[t, a] == 1:
                    actores.append(a)
        # calculamos el costo en base a la cantidad de actores
        actores = set(actores)
        costo += len(actores)
                
    # RESTRICCION: calcular un costo muy alto si se excede el maximo de 6 tomas
    for d in range(tomas):
        if len(dias[d]) > 6:
            costo += COSTO_RESTRICCION
    
    return costo    

In [339]:
# función para imprimir el detalle de tomas por cada día
def imprimir_solucion(problema, solucion):
    dias = {}
    # recorremos las tomas y las agrupamos por dia
    for i in range(len(solucion)):
        if not solucion[i] in dias:
            dias[solucion[i]] = []
        dias[solucion[i]].append(i + 1) # así las tomas empiezan en 1
    # listamos las tomas por cada dia
    tomas = list(dias.values())
    for i in range(len(tomas)):
        # actores que van a participar en las tomas de ese dia
        actores = []
        # recorremos las tomas de ese día 
        for t in tomas[i]:
            for a in range(problema.shape[1]):
                if problema[t-1, a] == 1: # t-1 porque las tomas ahora arrancan en 1
                    actores.append(a + 1) # asi los actores empiezan en 1
        actores = list(set(actores))
        print(f"día {i+1}: Tomas = {tomas[i]}, Actores = {actores} ({len(actores)})")
    print(f"Costo de la solucion: { calcular_costo(problema, solucion) }")

Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

### Algoritmo Genético

In [331]:
#Genera una poblacion inicial de soluciones de tamaño N.
def generar_poblacion(problema,N):
    return [np.random.randint(0, problema.shape[0], problema.shape[0]) for _ in range(N)]

In [332]:
#Evalua la población y devuelve el mejor individuo
def Evaluar_Poblacion(poblacion, problem):
    solucion = None
    costo = np.iinfo(np.int32).max
    for s in poblacion:
        c = calcular_costo(problema, s)
        if c < costo:
            solucion = s
            costo = c
    return solucion, costo

In [333]:
# Funcion de cruce. Recibe una poblacion(lista de soluciones) y devuelve la población ampliada con los hijos.
# Todos los individuos de la población son selecionados para el cruce(si la población es par)
def Cruzar(poblacion, problem, mutacion):
    soluciones = []
    soluciones += poblacion
    # los desordenamos para que el cruce sea al azar
    np.random.shuffle(poblacion)
    for p in range(0, len(poblacion) - 1, 2):
        hijo = Descendencia([poblacion[p], poblacion[p+1]], problem, mutacion)
        soluciones.append(hijo)
    return soluciones

In [334]:
# Funcion para generar hijos a partir de 2 padres:
# Se elige el metodo de 1-punto de corte
# Se aplica mutacion
def Descendencia(padres, problem, mutacion):
    # Se elige el metodo de 1-punto de corte
    punto = np.random.randint(0, len(padres[0]))
    hijo = np.concatenate([padres[0][:punto], padres[1][punto:]])
    # se aplica mutacion
    hijo = Mutar(hijo, mutacion)
    return hijo

In [335]:
# Funcion de mutación. Al azar se aplica alguna de estas operaciones:
# - se elije una toma al azar y se le asigna un día al azar
# - se intercambian los dias de dos tomas
# Se hace mutaciones mutacion% de las veces, si no hay mutacion se retorna None
def Mutar(solucion, mutacion):
    # mutacion solo el % de la veces
    if mutacion > np.random.uniform():
        operacion = np.random.randint(0, 2)
        if operacion == 0:
            # se elije una toma al azar y se le asigna un día al azar
            toma = np.random.randint(0, len(solucion))
            solucion[toma] = np.random.randint(0, len(solucion))
        elif operacion == 1:
            # se intercambian los dias de dos tomas
            t1, t2 = np.random.randint(0, len(solucion)), np.random.randint(0, len(solucion))
            solucion[t1], solucion[t2] = solucion[t2], solucion[t1]
    return solucion

In [336]:
#Funcion de seleccion de la población. Recibe como parametro una poblacion y
# devuelve una poblacion a la que se ha eliminado individuos poco aptos (fitness alto) y para mantener una poblacion estable de N individuos
# Se tiene en cuenta el porcentaje elitismo pasado como parametro
# Para los individuos que no son de la elite, se tomará un 10% al azar (asi damos chance a sobrevivir a uno muy malo y tenemos mas diversidad) 
# y se usará una selección de ruleta(proporcional a su fitness) con el resto.
def Seleccionar(poblacion, problema, N, elitismo):
    soluciones = []

    # calcular los costos y ordenar
    costos = [calcular_costo(problema, solucion) for solucion in poblacion]
    ranking = zip(poblacion, costos)
    ranking = sorted(ranking, key=lambda item: item[1])

    # salvamos a la elite
    elitismo = min(int(len(ranking) * elitismo), N)
    for i in range(elitismo):
        s = ranking.pop(0)
        soluciones.append(s[0])
        
    # se salvan el 10% al azar
    salvar = min(int(len(ranking) * .1), N - len(solucion))
    for i in range(salvar):
        s = np.random.randint(0, len(ranking))
        s = ranking.pop(s)
        soluciones.append(s[0])

    #vamos salvando hasta llegar a N
    # se usará una selección de ruleta(proporcional a su fitness) con el resto.
    max_costo = ranking[-1][1]
    min_costo = ranking[0][1]
    while(len(ranking) > 0 and len(soluciones) < N):
        keep = []
        for i in range(len(ranking)):
            s = ranking[i]
            prob = (s[1] - min_costo) / (max_costo - min_costo)
            if prob < np.random.uniform():
                # se salva
                soluciones.append(s[0])
                if (len(soluciones) == N):
                    break
            else:
                # sigue participando
                keep.append(s)
        ranking = keep
    
    return soluciones


In [337]:
#Funcion principal del algoritmo genetico
#######################################################3
def algoritmo_genetico(problema=problema, N=100, mutacion=.15, elitismo=.1, generaciones=100):
    # problem = datos del problema
    # N = Tamaño de la población
    # mutacion = probabilidad de una mutación
    # elitismo = porcion de la mejor poblacion a mantener
    # generaciones = nº de generaciones a generar para finalizar

    #Genera la poblacion inicial
    poblacion = generar_poblacion(problema, N)

    #Inicializamos valores para la mejor solucion
    (mejor_solucion, mejor_costo) = Evaluar_Poblacion(poblacion, problema)

    #Condicion de parada
    parar = False
    n=0
    #Inciamos el cliclo de generaciones
    while(parar == False) :
        #Cruce de la poblacion(incluye mutación)
        poblacion = Cruzar(poblacion, problema, mutacion)
        
        #Seleccionamos la población
        poblacion = Seleccionar(poblacion, problema, N, elitismo)
        
        #Evaluamos la nueva población
        (mejor_solucion, mejor_costo) = Evaluar_Poblacion(poblacion, problema)
        
        #print("Generacion #", n, "\nLa mejor solución es:" , mejor_solucion, "\ncon costo " , mejor_costo, "\n")
        
        #Numero de generaciones. Criterio de parada
        if n==generaciones:
            parar = True
        n +=1

    return mejor_solucion

In [341]:
# obtener una solucion usando algoritmo genetico
solucion_genetico = algoritmo_genetico(problema=problema,N=1000, mutacion=.3, elitismo=.4, generaciones=200)
# finalmente imprimir 
imprimir_solucion(problema, solucion_genetico)

día 1: Tomas = [1, 2, 5, 11, 19, 23], Actores = [1, 2, 3, 4, 5, 8] (6)
día 2: Tomas = [3, 4, 8, 10, 21, 29], Actores = [1, 2, 5, 6, 7, 8, 9] (7)
día 3: Tomas = [6, 7, 13, 20, 26, 27], Actores = [1, 2, 3, 4, 5, 9] (6)
día 4: Tomas = [9, 15, 16, 25, 28, 30], Actores = [1, 2, 4, 7, 10] (5)
día 5: Tomas = [12, 14, 17, 18, 22, 24], Actores = [1, 2, 3, 4, 6] (5)
Costo de la solucion: 29


(*)Calcula la complejidad del algoritmo

Respuesta

Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

Aplica el algoritmo al juego de datos generado

Respuesta

Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta